# Solution 06 — aggregation & anomaly detection

Reference for the aggregating consumer.

In [ ]:
from confluent_kafka import Consumer
from collections import defaultdict
from datetime import datetime
from IPython.display import clear_output
import json, time

BROKER = 'redpanda:29092'
MAX_SPEED_KMH = 80.0

consumer = Consumer({
    'bootstrap.servers': BROKER,
    'group.id':          'rider-stats',
    'auto.offset.reset': 'earliest',
})
consumer.subscribe(['rider-locations'])

## Step 1 — aggregate

In [ ]:
totals    = defaultdict(float)
counts    = defaultdict(int)
mins      = defaultdict(lambda: float('inf'))
maxs      = defaultdict(lambda: float('-inf'))
anomalies = []

messages_read, empty_polls = 0, 0
while messages_read < 200 and empty_polls < 5:
    msg = consumer.poll(2.0)
    if msg is None: empty_polls += 1; continue
    if msg.error(): continue
    empty_polls = 0; messages_read += 1

    try:
        data  = json.loads(msg.value().decode())
        rider = data.get('rider_id', 'unknown')
        speed = float(data.get('speed_kmh', 0.0))

        totals[rider] += speed
        counts[rider] += 1
        mins[rider]    = min(mins[rider], speed)
        maxs[rider]    = max(maxs[rider], speed)

        if speed > MAX_SPEED_KMH:
            anomalies.append({
                'rider': rider, 'speed': speed,
                'offset': msg.offset(), 'partition': msg.partition(),
            })
    except Exception as e:
        print(f'Parse error: {e}')
print(f'Processed {messages_read} events.')

## Step 2 — summary table

In [ ]:
print(f'{"Rider":<7} | {"Count":>5} | {"Min":>10} | {"Max":>10} | {"Avg":>10}')
print('-' * 54)
for rider in sorted(totals.keys()):
    n   = counts[rider]
    avg = totals[rider] / n
    print(f'{rider:<7} | {n:>5} | '
          f'{mins[rider]:>5.1f} km/h | '
          f'{maxs[rider]:>5.1f} km/h | '
          f'{avg:>5.1f} km/h')

## Step 3 — anomaly report

The glitches do pull up the average. A dashboard would usually drop flagged pings before averaging, but keep counting them, so that a phone with a bad GPS shows up.

In [ ]:
if not anomalies:
    print('No anomalies. Run the anomaly simulator in exercise_05_produce_stream.')
else:
    print(f'{len(anomalies)} anomalies\n')
    for a in anomalies:
        pct = (a['speed'] / MAX_SPEED_KMH - 1) * 100
        print(f'{a["rider"]:>7} | {a["speed"]:>6.1f} km/h | P{a["partition"]} '
              f'offset={a["offset"]} ({pct:+.0f}%)')

## Bonus — live aggregation

Run this with the streaming producer (`exercise_05_produce_stream`) going in another tab. The display refreshes every 10 events so you see the running averages stabilise as more data arrives. The cell stops by itself after `LIVE_SECONDS`, or earlier when you press ■.

**Stuck riders.** A per-event threshold cannot detect a rider who stands still for 15 minutes: you have to remember, per rider, since when the speed has been 0, and raise the alert when no movement arrived for long enough. That is per-key state over *time* — exactly what Flink's windows and timers are made for.

In [ ]:
LIVE_SECONDS = 300

totals_live    = defaultdict(float)
counts_live    = defaultdict(int)
anomaly_count  = 0
total_received = 0

deadline = time.time() + LIVE_SECONDS
try:
    while time.time() < deadline:
        msg = consumer.poll(0.5)
        if msg is None or msg.error(): continue
        try:
            data  = json.loads(msg.value().decode())
            rider = data.get('rider_id', 'unknown')
            speed = float(data.get('speed_kmh', 0.0))
            totals_live[rider] += speed
            counts_live[rider] += 1
            total_received     += 1
            if speed > MAX_SPEED_KMH:
                anomaly_count += 1
        except Exception:
            continue

        if total_received % 10 == 0:
            clear_output(wait=True)
            ts = datetime.now().strftime('%H:%M:%S')
            print(f'{total_received} events  [{ts}]  anomalies: {anomaly_count}')
            print(f'{"Rider":<7} | {"Count":>5} | {"Avg":>10}')
            print('-' * 28)
            for r in sorted(totals_live.keys()):
                n   = counts_live[r]
                avg = totals_live[r] / n
                print(f'{r:<7} | {n:>5} | {avg:>5.1f} km/h')
    print(f'Time is up. {total_received} events received.')
except KeyboardInterrupt:
    print('Stopped.')
finally:
    consumer.close()